In [ ]:
# Simple RAG Agent

# Load the pdf files and split them into chunks

In [1]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader

directory_loader = DirectoryLoader("data", glob="**/*.pdf", loader_cls=PyMuPDFLoader)

docs = directory_loader.load()

print(docs) 

[Document(metadata={'producer': 'macOS Version 15.4.1 (Build 24E263) Quartz PDFContext, AppendMode 1.1', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-09-12T20:05:32+00:00', 'source': 'data/howpeopleuseai.pdf', 'file_path': 'data/howpeopleuseai.pdf', 'total_pages': 64, 'format': 'PDF 1.6', 'title': 'How People Use ChatGPT', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-09-15T10:32:36-04:00', 'trapped': '', 'modDate': "D:20250915103236-04'00'", 'creationDate': 'D:20250912200532Z', 'page': 0}, page_content='NBER WORKING PAPER SERIES\nHOW PEOPLE USE CHATGPT\nAaron Chatterji\nThomas Cunningham\nDavid J. Deming\nZoe Hitzig\nChristopher Ong\nCarl Yan Shan\nKevin Wadman\nWorking Paper 34255\nhttp://www.nber.org/papers/w34255\nNATIONAL BUREAU OF ECONOMIC RESEARCH\n1050 Massachusetts Avenue\nCambridge, MA 02138\nSeptember 2025\nWe acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan \nBeiermeister, Rachel Brown, Cassandra Duchan Solis, Jason Kwon,

In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=100)
split_docs = text_splitter.split_documents(docs)

print(split_docs)

len(split_docs)

[Document(metadata={'producer': 'macOS Version 15.4.1 (Build 24E263) Quartz PDFContext, AppendMode 1.1', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-09-12T20:05:32+00:00', 'source': 'data/howpeopleuseai.pdf', 'file_path': 'data/howpeopleuseai.pdf', 'total_pages': 64, 'format': 'PDF 1.6', 'title': 'How People Use ChatGPT', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-09-15T10:32:36-04:00', 'trapped': '', 'modDate': "D:20250915103236-04'00'", 'creationDate': 'D:20250912200532Z', 'page': 0}, page_content='NBER WORKING PAPER SERIES\nHOW PEOPLE USE CHATGPT\nAaron Chatterji\nThomas Cunningham\nDavid J. Deming\nZoe Hitzig\nChristopher Ong\nCarl Yan Shan\nKevin Wadman\nWorking Paper 34255\nhttp://www.nber.org/papers/w34255\nNATIONAL BUREAU OF ECONOMIC RESEARCH\n1050 Massachusetts Avenue\nCambridge, MA 02138\nSeptember 2025\nWe acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan \nBeiermeister, Rachel Brown, Cassandra Duchan Solis, Jason Kwon,

200

# Create embeddings

In [3]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [4]:
# create Qdrant Vector Store
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance,VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="rag_collection_name",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="rag_collection_name",
    embedding=embeddings,
)

In [5]:
# Add documents to the vector store
vector_store.add_documents(documents=split_docs)

['f2b46e02ccb541708d1cde56d0a2f37c',
 'e13ef289404f44bb986dec8da8682768',
 'b8e21594e83145ddb9dee04f6a315a5e',
 '3b04dd0752a9451a8e87b5bf85b8686a',
 '6ac6e1fe7ec9461badb8c25c3df165a4',
 '727ec63c456d44bbbc2bb96ae0d2d2e9',
 '619c9771c8a945c6b8177ff2ec5c3eb6',
 '3c0b9b65e16a4cc393b961ad237c653f',
 'ba5d22fa47f44339b4a828fdcd3e4f0f',
 '66b3bf9f6ee141ac8cf2923fcf7bdcc3',
 '96d6b69291354c2787ca317e2e7329a0',
 '0004ca79a9a8456190152f2f9f49fbd5',
 'bfdefd26cc584f55b8f70457db3b8445',
 'a04745d00d8b406faa3397b30cba2991',
 '38cf02f19c4e4f34a622378b4b517432',
 '7be6d8836a254c46af42994254cc21dd',
 'f67f700ee5554b45bdf048419abee92c',
 '3e0c247a7b7549f48427748529011211',
 '26285c514eb846d58a75eb896f184415',
 '3672870ca6d74fcb886c6f5bf9efc5cf',
 'e144e2510f20445792844a7c26a136f3',
 '13188c47407947bca9beaa4f41195911',
 'd5ac4032797e4dafa0144c7f2e936080',
 '8a56b2574483452ca760015496589399',
 '18dd442db0b844d38f302a2e2a793ce7',
 '89ecac2f97e54b5ab0f5682a85181bd1',
 '52365729f863430d8b9d3db26504a61b',
 

In [6]:
# Lets create a retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [7]:
retriever.invoke("what is the main purpose of the document?")

[Document(metadata={'producer': 'macOS Version 15.4.1 (Build 24E263) Quartz PDFContext, AppendMode 1.1', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-09-12T20:05:32+00:00', 'source': 'data/howpeopleuseai.pdf', 'file_path': 'data/howpeopleuseai.pdf', 'total_pages': 64, 'format': 'PDF 1.6', 'title': 'How People Use ChatGPT', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-09-15T10:32:36-04:00', 'trapped': '', 'modDate': "D:20250915103236-04'00'", 'creationDate': 'D:20250912200532Z', 'page': 21, '_id': '70611eec8f6e4a74817251620284ed19', '_collection_name': 'rag_collection_name'}, page_content='at work appears to be focused on two broad functions: 1) obtaining, documenting, and interpreting\ninformation; and 2) making decisions, giving advice, solving problems, and thinking creatively.\n20'),
 Document(metadata={'producer': 'macOS Version 15.4.1 (Build 24E263) Quartz PDFContext, AppendMode 1.1', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-09-12T20:05:3

In [8]:
# produce node for the retrieval
def retrieve(state):
    retrieved_docs = retriever.invoke(state["question"])
    return {"context": retrieved_docs}

In [28]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
    You are a helpful assistant who can answer questions based on the following context only:
    If you cannot answer the question based on the context - you must say "I don't know".
    
    ### Context: 
    {context}
    
    ### Question: 
    {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

In [29]:
# Create LLM instance
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4.1-mini")

In [30]:
# Create a generator
def generate(state):
    docs_content="\n\n".join( doc.page_content for doc in state["context"])
    message = rag_prompt.format_messages(question=state["question"], context=docs_content)
    response = llm.invoke(message)
    return {"response": response.content}

In [14]:
# build a state 
from typing_extensions import TypedDict, List
from langchain_core.documents import Document

class State(TypedDict):
    question: str
    context: List[Document]
    response: str

In [15]:
# Lets build a graph 
from langgraph.graph import START,StateGraph

# 
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph=graph_builder.compile()

In [31]:
# Run the graph
response = graph.invoke({"question": "how do people use chatgpt?"})

print(response["response"])


People use ChatGPT in various ways that can be broadly classified into three categories according to a taxonomy introduced by the authors: Asking, Doing, and Expressing. 

- **Asking:** Users seek information or clarification to inform decisions, corresponding to problem-solving and knowledge work.
- **Doing:** Although not fully detailed in the provided context, this category typically involves users asking ChatGPT to perform tasks or generate specific outputs.
- **Expressing:** This likely involves using ChatGPT for communication, creativity, or sharing ideas, though specific details are not included in the excerpt.

Additionally, ChatGPT usage varies by demographics and purposes: 
- It is more frequently used by men, young people, and those with tertiary or graduate education, although the gender gap is narrowing.
- Usage has grown faster in low- and middle-income countries.
- Educated users and those in highly-paid professional occupations are more likely to use ChatGPT for work-re

In [18]:
# Lets create some tools 

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.arxiv.tool import ArxivQueryRun

arxiv_tool = ArxivQueryRun()
tavily_tool = TavilySearchResults(max_results=5)

/var/folders/0j/52nrhqqd6hq140jsz5rrd5km0000gq/T/ipykernel_84315/3746444787.py:7: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5)


 ## Note

 `ai_rag_tool` returns JSON with BOTH:
- `answer`: The generated response
- `contexts`: List of retrieved document chunks

This allows RAGAS to evaluate whether the answer is grounded in the retrieved contexts.


In [19]:
# lets create a tool for the graph  
from langchain_core.tools import tool
import json

@tool
def ai_rag_tool(question: str):
    """
    Use this tool to answer questions based on the context provided. Input should be a fully formed question.
    """
    
    response = graph.invoke({"question": question})
    
    # Return BOTH the answer AND the retrieved contexts
    # Format: JSON string with answer and contexts
    result = {
        "answer": response["response"],
        "contexts": [doc.page_content for doc in response["context"]]
    }
    
    return json.dumps(result)


In [20]:
# Test the updated ai_rag_tool to verify it returns contexts
test_result = ai_rag_tool.invoke({"question": "How do people use chatgpt?"})
print("Tool result:", test_result[:200], "...")

# Parse it to see the structure
import json
parsed = json.loads(test_result)
print("\nParsed structure:")
print(f"- answer: {parsed['answer'][:100]}...")
print(f"- contexts: {len(parsed['contexts'])} chunks retrieved")
print(f"- First context: {parsed['contexts'][0][:100]}...")


Tool result: {"answer": "People use ChatGPT in various ways, which can be broadly classified into three categories based on the type of output the user is seeking: Asking, Doing, or Expressing. \n\n- **Asking** in ...

Parsed structure:
- answer: People use ChatGPT in various ways, which can be broadly classified into three categories based on t...
- contexts: 5 chunks retrieved
- First context: How People Use ChatGPT
Aaron Chatterji, Thomas Cunningham, David J. Deming, Zoe Hitzig, Christopher ...


In [21]:
# lets create a tool belt

tools = [ai_rag_tool, arxiv_tool, tavily_tool]

In [35]:
# Bind tools to the llm
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1",temperature=0)
model=model.bind_tools(tools)

model.invoke("How do people use chatgpt? Use the tools to answer the question")

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_6kH3UstGZlKey4J6lX71BZxr', 'function': {'arguments': '{"question":"How do people use ChatGPT?"}', 'name': 'ai_rag_tool'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 203, 'total_tokens': 225, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_422e2d36a8', 'id': 'chatcmpl-CSq5SwMExy85AxQ9qQllUAuy1ucSv', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--a4d88551-220c-4635-a5cd-9b729cdefb14-0', tool_calls=[{'name': 'ai_rag_tool', 'args': {'question': 'How do people use ChatGPT?'}, 'id': 'call_6kH3UstGZlKey4J6lX71BZxr', 'type': 'tool_call'}], usage_metadata={'input_tokens': 203, 'output_tokens': 22, 't

In [36]:
# Langgraph Agent

from typing import TypedDict, Annotated,List
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    messages:Annotated[list,add_messages]
    context:List[Document]


In [37]:
from langgraph.prebuilt import ToolNode

def call_model(state):
    messages = state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

In [38]:
from langgraph.graph import StateGraph,END

uncompiled_graph= StateGraph(AgentState)
uncompiled_graph.add_node("agent",call_model)
uncompiled_graph.add_node("action",tool_node)




In [39]:
# add function/runnable for the conditional edge

def should_continue(state):
    last_message=state["messages"][-1]
    if last_message.tool_calls:
        return "action"
    return END 


uncompiled_graph.set_entry_point("agent")
uncompiled_graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "action": "action",  # when function returns "action"
        END: END             # when function returns END
    }
)

uncompiled_graph.add_edge("action","agent");
    

In [40]:
compiled_graph=uncompiled_graph.compile()


In [43]:
# Run the graph
from langchain_core.messages import HumanMessage

inputs={"messages":[HumanMessage(content="How do people use chatgpt? Answer the question using the tools")]}

from langchain_core.messages import ToolMessage, HumanMessage

async for chunk in compiled_graph.astream(inputs, stream_mode="updates"):
    for node, update in chunk.items():
        print(f"\n📡 Receiving updates from node: {node}")

        # Safely handle presence of "messages"
        for msg in update.get("messages", []):
            
            # 🧰 Tool message (result of a tool call)
            if isinstance(msg, ToolMessage):
                print(f"  🧰 ToolMessage from: {msg.name}")
                print(f"  🧾 Content: {msg.content}")
                print(f"  🔖 Tool Call ID: {msg.tool_call_id}")

            # 👤 Human message (user input)
            elif isinstance(msg, HumanMessage):
                print("  👤 HumanMessage:")
                print(f"  💬 {msg.content}")

            # fallback for other message types (AIMessage, etc.)
            else:
                print(f"  ⚙️ Other message type: {type(msg).__name__}")
                print(f"  💬 {getattr(msg, 'content', msg)}")
    


📡 Receiving updates from node: agent
  ⚙️ Other message type: AIMessage
  💬 

📡 Receiving updates from node: action
  🧰 ToolMessage from: ai_rag_tool
  🧾 Content: {"answer": "People use ChatGPT primarily for modifying user text, such as editing, critiquing, and translating, which accounts for about two-thirds of all Writing messages. Around 10% of all messages involve requests for tutoring or teaching, indicating that education is a key use case. A smaller share of messages\u20144.2%\u2014relate to computer programming, and only 1.9% concern companionship or social-emotional issues. Overall, users employ ChatGPT for a variety of purposes categorized as Asking (seeking information or clarification), Doing (modifying or creating content), and Expressing.", "contexts": ["How People Use ChatGPT\nAaron Chatterji, Thomas Cunningham, David J. Deming, Zoe Hitzig, Christopher Ong, Carl\nYan Shan, and Kevin Wadman\nNBER Working Paper No. 34255\nSeptember 2025\nJEL No. J01, O3, O4\nABSTRACT\nDes

# Ragas Baseline And SDG

In [44]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())



In [45]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

dataset = generator.generate_with_langchain_docs(docs, testset_size=9)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/40 [00:00<?, ?it/s]

Property 'summary' already exists in node 'e4f0a3'. Skipping!
Property 'summary' already exists in node '86c731'. Skipping!
Property 'summary' already exists in node '258a06'. Skipping!
Property 'summary' already exists in node 'd89984'. Skipping!
Property 'summary' already exists in node 'a589a1'. Skipping!
Property 'summary' already exists in node '1b4c61'. Skipping!
Property 'summary' already exists in node 'e3d7e8'. Skipping!
Property 'summary' already exists in node '9fe11a'. Skipping!
Property 'summary' already exists in node '3f798a'. Skipping!
Property 'summary' already exists in node '9aef83'. Skipping!
Property 'summary' already exists in node '20fd16'. Skipping!
Property 'summary' already exists in node '0d9491'. Skipping!
Property 'summary' already exists in node 'e461c1'. Skipping!
Property 'summary' already exists in node '62525d'. Skipping!
Property 'summary' already exists in node '7fd69f'. Skipping!
Property 'summary' already exists in node '5c48ba'. Skipping!
Property

...

Property 'summary_embedding' already exists in node 'd89984'. Skipping!
Property 'summary_embedding' already exists in node '86c731'. Skipping!
Property 'summary_embedding' already exists in node '258a06'. Skipping!
Property 'summary_embedding' already exists in node 'e4f0a3'. Skipping!
Property 'summary_embedding' already exists in node 'e3d7e8'. Skipping!
Property 'summary_embedding' already exists in node '9fe11a'. Skipping!
Property 'summary_embedding' already exists in node 'a589a1'. Skipping!
Property 'summary_embedding' already exists in node '3f798a'. Skipping!
Property 'summary_embedding' already exists in node '1b4c61'. Skipping!
Property 'summary_embedding' already exists in node 'fa4413'. Skipping!
Property 'summary_embedding' already exists in node '20fd16'. Skipping!
Property 'summary_embedding' already exists in node '9aef83'. Skipping!
Property 'summary_embedding' already exists in node 'e461c1'. Skipping!
Property 'summary_embedding' already exists in node 'f9f5d2'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [46]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What Acemoglu 2024 say about AI effect on econ...,[Introduction ChatGPT launched in November 202...,Acemoglu (2024) discusses the intensified inte...,single_hop_specifc_query_synthesizer
1,How many 700 million users were sending messag...,[Introduction ChatGPT launched in November 202...,"By July 2025, 700 million users were sending 1...",single_hop_specifc_query_synthesizer
2,how much money us users get from generative ai...,[Conclusion This paper studies the rapid growt...,Collis and Brynjolfsson (2025) estimate that U...,single_hop_specifc_query_synthesizer
3,How has ChatGPT usage changed recently in low-...,[Conclusion This paper studies the rapid growt...,ChatGPT usage has grown especially fast over t...,single_hop_specifc_query_synthesizer
4,How do the ChatGPT adoption and usage statisti...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,ChatGPT launched in November 2022 and by July ...,multi_hop_abstract_query_synthesizer
5,How does the rapid ChatGPT adoption and usage ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,ChatGPT launched in November 2022 and by July ...,multi_hop_abstract_query_synthesizer
6,how ChatGPT adoption and usage statistics show...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,ChatGPT launched in November 2022 and by July ...,multi_hop_abstract_query_synthesizer
7,How did ChatGPT's user base and message volume...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT had grown rapidly since ...",multi_hop_specific_query_synthesizer
8,How did the launch of ChatGPT in November 2022...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,ChatGPT launched in November 2022 and experien...,multi_hop_specific_query_synthesizer
9,How did the launch of ChatGPT in November 2022...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,ChatGPT launched in November 2022 and experien...,multi_hop_specific_query_synthesizer


# Code to extract context from the tool calls

In [53]:
import json
import re
from typing import Any, List, Dict, Optional

class ContextParser:
    """Parser to extract contexts from tool call results in agent messages."""
    
    @staticmethod
    def detect_tool_type(message: Any) -> Optional[str]:
        """Detect which tool was called based on the message."""
        try:
            # Check if message has tool_calls attribute
            if hasattr(message, 'tool_calls') and message.tool_calls:
                tool_name = message.tool_calls[0].get('name', '')
                if 'tavily' in tool_name.lower():
                    return 'tavily'
                elif 'arxiv' in tool_name.lower():
                    return 'arxiv'
                elif 'ai_rag' in tool_name.lower() or 'rag' in tool_name.lower():
                    return 'ai_rag'
            
            # Check if it's a ToolMessage with a name attribute
            if hasattr(message, 'name'):
                tool_name = message.name
                if 'tavily' in tool_name.lower():
                    return 'tavily'
                elif 'arxiv' in tool_name.lower():
                    return 'arxiv'
                elif 'ai_rag' in tool_name.lower() or 'rag' in tool_name.lower():
                    return 'ai_rag'
                    
        except Exception as e:
            print(f"Error detecting tool type: {e}")
        return None
    
    @staticmethod
    def extract_content(message: Any) -> Optional[str]:
        """Extract content from a message object."""
        try:
            if hasattr(message, 'content'):
                content = message.content
                if isinstance(content, str):
                    # Handle JSON-like strings with quotes
                    if content.startswith('content="') or content.startswith("content='"):
                        # Find the matching end quote by counting quotes
                        stack = []
                        in_quote = False
                        for i, char in enumerate(content):
                            if char == '"' and (i == 0 or content[i-1] != '\\'):
                                if not in_quote:
                                    stack.append(i)
                                    in_quote = True
                                else:
                                    stack.pop()
                                    if not stack and 'name=' in content[i:]:
                                        return content[8:i]  # 8 is len('content="')
                    return content
        except Exception as e:
            print(f"Error extracting content: {e}")
        return None
    
    @staticmethod
    def parse_tavily_results(content: str) -> List[Dict]:
        """Parse Tavily search results from content string."""
        try:
            # Parse the JSON string
            if isinstance(content, str):
                data = json.loads(content)
                
                # Get results array from the appropriate location
                if isinstance(data, list):
                    results = data
                elif isinstance(data, dict):
                    if 'artifact' in data and 'results' in data['artifact']:
                        results = data['artifact']['results']
                    elif 'results' in data:
                        results = data['results']
                    else:
                        results = []
                else:
                    results = []
                
                # Ensure content is string in each result
                for result in results:
                    if isinstance(result.get('content'), dict):
                        result['content'] = str(result['content'])
                        
                return results
        except Exception as e:
            print(f"Error parsing Tavily results: {str(e)}")
        return []
    
    @staticmethod
    def parse_ai_rag_results(content: str) -> List[Dict]:
        """Parse AI RAG tool results from content string."""
        try:
            # The ai_rag_tool now returns JSON with 'answer' and 'contexts'
            data = json.loads(content)
            
            # Extract contexts list
            if isinstance(data, dict) and 'contexts' in data:
                contexts = data['contexts']
                # Convert to list of dicts with 'content' key for consistency
                documents = [{'content': ctx} for ctx in contexts if ctx]
                return documents
            
            # Fallback: if it's just a plain string (old format)
            elif isinstance(content, str) and content.strip():
                return [{'content': content}]
            
            return []
        except json.JSONDecodeError:
            # If it's not JSON, treat it as plain text
            print("ai_rag content is not JSON, treating as plain text")
            return [{'content': content}] if content else []
        except Exception as e:
            print(f"Error parsing AI RAG results: {str(e)}")
            return []
    
    @staticmethod  
    def parse_arxiv_results(content: str) -> List[Dict]:
        """Parse arXiv results from content string."""
        try:
            papers = []
            # Split by "Published: " but keep the delimiter
            sections = re.split(r'(?=Published: )', content)
            
            for section in sections:
                if not section.strip():
                    continue
                
                # Extract paper details
                date_match = re.search(r'Published: (\d{4}-\d{2}-\d{2})', section)
                title_match = re.search(r'Title: (.*?)(?=\nAuthors:|$)', section, re.DOTALL)
                authors_match = re.search(r'Authors: (.*?)(?=\nSummary:|$)', section, re.DOTALL)
                summary_match = re.search(r'Summary: (.*?)(?=\n\nPublished:|$)', section, re.DOTALL)
                
                if date_match and title_match:
                    summary = summary_match.group(1).strip() if summary_match else ''
                    # Ensure summary is a string
                    if isinstance(summary, dict):
                        summary = str(summary)
                    
                    paper = {
                        'date': date_match.group(1),
                        'title': title_match.group(1).strip(),
                        'authors': authors_match.group(1).strip() if authors_match else '',
                        'summary': summary
                    }
                    papers.append(paper)
            
            return papers
        except Exception as e:
            print(f"Error parsing arXiv results: {str(e)}")
        return []


def parse_tool_call(message: Any) -> List[Dict]:
    """Parse tool call results from a conversation message."""
    parser = ContextParser()
    
    print("\n=== Starting message parsing ===")
    print(f"Message type: {type(message)}")
    
    # Detect tool type
    tool_type = parser.detect_tool_type(message)
    print(f"Detected tool type: {tool_type}")
    
    if tool_type is None:
        print("Could not detect tool type")
        return []
    
    # Extract content
    content = parser.extract_content(message)
    if content is None:
        print("Could not extract content")
        return []
    
    print(f"Extracted content type: {type(content)}")
    
    # Parse based on tool type
    if tool_type == 'tavily':
        results = parser.parse_tavily_results(content)
        print(f"Parsed {len(results)} Tavily results")
        return results
    elif tool_type == 'ai_rag':
        results = parser.parse_ai_rag_results(content)
        print(f"Parsed {len(results)} AI RAG results")
        return results
    elif tool_type == 'arxiv':
        results = parser.parse_arxiv_results(content)
        print(f"Parsed {len(results)} arXiv results")
        return results
    
    return []


In [52]:
def extract_contexts_from_messages(messages: List) -> List[str]:
    """
    Extract all contexts from a list of messages in a conversation.
    This looks for ToolMessage objects and parses their content.
    """
    all_contexts = []
    
    for message in messages:
        # Check if it's a ToolMessage (contains tool call results)
        if hasattr(message, 'name') and hasattr(message, 'content'):
            print(f"\nProcessing ToolMessage from {message.name}")
            
            # Parse the tool call results
            parsed_results = parse_tool_call(message)
            
            # Extract content from parsed results
            for result in parsed_results:
                if isinstance(result, dict):
                    # Different tools return different formats
                    if 'content' in result:
                        all_contexts.append(result['content'])
                    elif 'summary' in result:
                        all_contexts.append(result['summary'])
                    elif 'text' in result:
                        all_contexts.append(result['text'])
    
    return all_contexts


def get_contexts_from_graph_response(response: dict) -> List[str]:
    """
    Extract contexts from a graph response that contains messages.
    """
    if 'messages' not in response:
        print("No messages found in response")
        return []
    
    contexts = extract_contexts_from_messages(response['messages'])
    print(f"\nExtracted {len(contexts)} total contexts")
    return contexts


In [54]:
# Updated evaluation loop with context extraction
for test_row in dataset:
    inputs = {"messages": [HumanMessage(content=test_row.eval_sample.user_input)]}
    response = compiled_graph.invoke(inputs)
    
    # Extract the response text from the last message
    test_row.eval_sample.response = response["messages"][-1].content
    
    # Extract contexts from tool calls in the conversation
    retrieved_contexts = get_contexts_from_graph_response(response)
    
    # Store contexts for evaluation
    test_row.eval_sample.retrieved_contexts = retrieved_contexts
    
    print(f"Question: {test_row.eval_sample.user_input[:100]}...")
    print(f"Retrieved {len(retrieved_contexts)} contexts")
    print("-" * 50)



Processing ToolMessage from None

=== Starting message parsing ===
Message type: <class 'langchain_core.messages.human.HumanMessage'>
Error detecting tool type: 'NoneType' object has no attribute 'lower'
Detected tool type: None
Could not detect tool type

Processing ToolMessage from None

=== Starting message parsing ===
Message type: <class 'langchain_core.messages.ai.AIMessage'>
Detected tool type: arxiv
Extracted content type: <class 'str'>
Parsed 0 arXiv results

Processing ToolMessage from arxiv

=== Starting message parsing ===
Message type: <class 'langchain_core.messages.tool.ToolMessage'>
Detected tool type: arxiv
Extracted content type: <class 'str'>
Parsed 3 arXiv results

Processing ToolMessage from None

=== Starting message parsing ===
Message type: <class 'langchain_core.messages.ai.AIMessage'>
Detected tool type: tavily
Extracted content type: <class 'str'>
Error parsing Tavily results: Expecting value: line 1 column 1 (char 0)
Parsed 0 Tavily results

Processing Tool

In [55]:
# Verify dataset is properly populated
df = dataset.to_pandas()
print("Dataset columns:", df.columns.tolist())
print("\nSample row check:")
sample = df.iloc[0]
print(f"- user_input: {type(sample['user_input'])} - {str(sample['user_input'])[:100]}...")
print(f"- response: {type(sample['response'])} - {str(sample['response'])[:100]}...")
print(f"- reference: {type(sample['reference'])} - {str(sample['reference'])[:100]}...")
print(f"- retrieved_contexts: {type(sample['retrieved_contexts'])} - Length: {len(sample['retrieved_contexts']) if sample['retrieved_contexts'] else 0}")

# Check if all required columns have data
print("\nData completeness check:")
print(f"- Rows with user_input: {df['user_input'].notna().sum()}/{len(df)}")
print(f"- Rows with response: {df['response'].notna().sum()}/{len(df)}")
print(f"- Rows with reference: {df['reference'].notna().sum()}/{len(df)}")
print(f"- Rows with retrieved_contexts: {df['retrieved_contexts'].notna().sum()}/{len(df)}")


Dataset columns: ['user_input', 'retrieved_contexts', 'reference_contexts', 'response', 'reference', 'synthesizer_name']

Sample row check:
- user_input: <class 'str'> - What Acemoglu 2024 say about AI effect on economy?...
- response: <class 'str'> - Daron Acemoglu’s 2024 work on the economic effects of AI, especially as summarized in his paper “The...
- reference: <class 'str'> - Acemoglu (2024) discusses the intensified interest in the effects of artificial intelligence on econ...
- retrieved_contexts: <class 'list'> - Length: 8

Data completeness check:
- Rows with user_input: 10/10
- Rows with response: 10/10
- Rows with reference: 10/10
- Rows with retrieved_contexts: 10/10


In [56]:
# Run RAGAS evaluation with properly populated dataset
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig
from ragas import EvaluationDataset
from ragas.llms import LangchainLLMWrapper

# Create evaluation dataset from pandas
evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

# Configure evaluation
custom_run_config = RunConfig(timeout=360)
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

# Run evaluation with all metrics
baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        LLMContextRecall(),      # Requires: user_input, retrieved_contexts, reference
        Faithfulness(),          # Requires: user_input, response, retrieved_contexts
        FactualCorrectness(),    # Requires: user_input, response, reference
        ResponseRelevancy(),     # Requires: user_input, response
        ContextEntityRecall(),   # Requires: user_input, retrieved_contexts, reference
        NoiseSensitivity()       # Requires: user_input, retrieved_contexts
    ],
    llm=evaluator_llm,
    run_config=custom_run_config
)

# Display results
print("RAGAS Evaluation Results:")
print(baseline_result)


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Exception raised in Job[35]: TimeoutError()


RAGAS Evaluation Results:
{'context_recall': 0.5893, 'faithfulness': 0.9107, 'factual_correctness': 0.4860, 'answer_relevancy': 0.9497, 'context_entity_recall': 0.3454, 'noise_sensitivity_relevant': 0.3869}


In [58]:
baseline_result.to_pandas()

,user_input,retrieved_contexts,reference_contexts,response,reference,context_recall,faithfulness,factual_correctness,answer_relevancy,context_entity_recall,noise_sensitivity_relevant
0,What Acemoglu 2024 say about AI effect on econ...,[Acemoglu and Johnson (2007) put forward the u...,[Introduction ChatGPT launched in November 202...,Daron Acemoglu’s 2024 work on the economic eff...,Acemoglu (2024) discusses the intensified inte...,1.000000,1.000000,0.06,0.941399,0.250000,0.035714
1,How many 700 million users were sending messag...,"[According to Brad Lightcap, OpenAI's Chief Op...",[Introduction ChatGPT launched in November 202...,"By July 2025, ChatGPT had over 700 million wee...","By July 2025, 700 million users were sending 1...",0.500000,1.000000,0.57,0.929332,0.600000,0.125000
2,how much money us users get from generative ai...,[educated and working in professional occupati...,[Conclusion This paper studies the rapid growt...,US users derive a surplus value of at least $9...,Collis and Brynjolfsson (2025) estimate that U...,1.000000,1.000000,0.75,0.920548,0.625000,0.400000
3,How has ChatGPT usage changed recently in low-...,"[But as of mid-2025, that gender gap was found...",[Conclusion This paper studies the rapid growt...,Recent studies and reports indicate that ChatG...,ChatGPT usage has grown especially fast over t...,1.000000,0.916667,0.47,0.986357,0.500000,0.583333
4,How do the ChatGPT adoption and usage statisti...,[### 5. How fast is ChatGPT growing in mobile ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,ChatGPT’s adoption and usage statistics highli...,ChatGPT launched in November 2022 and by July ...,0.142857,1.000000,0.39,0.933945,0.071429,0.553191
5,How does the rapid ChatGPT adoption and usage ...,[ChatGPT launched as a “research preview” in l...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT's adoption and usage sta...",ChatGPT launched in November 2022 and by July ...,0.714286,0.918033,0.53,0.939868,0.375000,NaN
6,how ChatGPT adoption and usage statistics show...,[Image 8: Profile picture of Nerdynav By Nerdy...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,Here’s a summary of ChatGPT’s adoption and usa...,ChatGPT launched in November 2022 and by July ...,0.250000,1.000000,0.40,0.957209,0.222222,0.719298
7,How did ChatGPT's user base and message volume...,[So the paper provides analysis of the chatbot...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT's user base and message ...","By July 2025, ChatGPT had grown rapidly since ...",0.285714,0.800000,0.49,0.988272,0.400000,0.480000
8,How did the launch of ChatGPT in November 2022...,[7\nConclusion\nThis paper studies the rapid g...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The launch of ChatGPT in November 2022 contrib...,ChatGPT launched in November 2022 and experien...,0.666667,1.000000,0.56,0.963721,0.210526,0.391304
9,How did the launch of ChatGPT in November 2022...,[7\nConclusion\nThis paper studies the rapid g...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The launch of ChatGPT in November 2022 was piv...,ChatGPT launched in November 2022 and experien...,0.333333,0.472222,0.64,0.936727,0.200000,0.194444


In [59]:
print(baseline_result)

{'context_recall': 0.5893, 'faithfulness': 0.9107, 'factual_correctness': 0.4860, 'answer_relevancy': 0.9497, 'context_entity_recall': 0.3454, 'noise_sensitivity_relevant': 0.3869}


## Let us use Cohere's Rerank model for re-Evaluating

Now that we've got our baseline - let's make a change and see how the model improves or doesn't improve!

In [68]:
# Let us build the adjusted retriever using cohere's rerank model

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance,VectorParams
from langchain_openai import OpenAIEmbeddings

text_splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=100)
split_docs = text_splitter.split_documents(docs)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")
client.create_collection(
    collection_name="cohere_rag_collection",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="cohere_rag_collection",
    embedding=embeddings,
)
vector_store.add_documents(documents=split_docs)
cohere_retriever = vector_store.as_retriever(search_kwargs={"k": 20})


from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

def cohere_adjusted_retriever(state):
  compressor = CohereRerank(model="rerank-v3.5",top_n=10)
  compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=cohere_retriever
  )
  retrieved_docs = compression_retriever.invoke(state["question"])
  return {"context" : retrieved_docs}



# Lets build a graph 
from langgraph.graph import START,StateGraph

# 
cohere_graph_builder = StateGraph(State).add_sequence([cohere_adjusted_retriever, generate])
cohere_graph_builder.add_edge(START, "cohere_adjusted_retriever")
cohere_graph=cohere_graph_builder.compile()

# lets create a tool for the graph  
from langchain_core.tools import tool
import json

@tool
def ai_rag_tool_cohere(question: str):
    """
    Use this tool to answer questions based on the context provided. Input should be a fully formed question.
    """
    
    response = cohere_graph.invoke({"question": question})
    
    # Return BOTH the answer AND the retrieved contexts
    # Format: JSON string with answer and contexts
    result = {
        "answer": response["response"],
        "contexts": [doc.page_content for doc in response["context"]]
    }
    
    return json.dumps(result)

tools_cohere = [ai_rag_tool_cohere, arxiv_tool, tavily_tool]

# Bind tools to the llm
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1",temperature=0)
model=model.bind_tools(tools_cohere)


from langgraph.prebuilt import ToolNode

def call_model(state):
    messages = state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}

tool_node_cohere = ToolNode(tools_cohere)

from langgraph.graph import StateGraph,END

cohere_uncompiled_graph= StateGraph(AgentState)
cohere_uncompiled_graph.add_node("agent",call_model)
cohere_uncompiled_graph.add_node("action",tool_node_cohere)

def should_continue(state):
    last_message=state["messages"][-1]
    if last_message.tool_calls:
        return "action"
    return END 


cohere_uncompiled_graph.set_entry_point("agent")
cohere_uncompiled_graph.add_conditional_edges(
    "agent", should_continue
)
cohere_uncompiled_graph.add_edge("action","agent");

cohere_compiled_graph=cohere_uncompiled_graph.compile()

# Run the graph
from langchain_core.messages import HumanMessage

inputs={"messages":[HumanMessage(content="How do people use chatgpt? Answer the question using the tools")]}

async for chunk in cohere_compiled_graph.astream(inputs,stream_mode="updates"):
    for node,values in chunk.items():
        print("Receiving updates from node",node)
        print(values["messages"])
        print("\n\n")




Receiving updates from node agent
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_rvxQpTxnLMvgYWt04v2gB12H', 'function': {'arguments': '{"question":"How do people use ChatGPT?"}', 'name': 'ai_rag_tool_cohere'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 204, 'total_tokens': 228, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_422e2d36a8', 'id': 'chatcmpl-CSrIjd5t5SZ57iEnBdVjwmTYsh7co', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--ddd6a0eb-43d4-4e7b-80a8-643f96494cbc-0', tool_calls=[{'name': 'ai_rag_tool_cohere', 'args': {'question': 'How do people use ChatGPT?'}, 'id': 'call_rvxQpTxnLMvgYWt04v2gB12H', 'type': 'tool_call'}], usage_metad

In [69]:
import time
import copy

rerank_dataset = copy.deepcopy(dataset)

# Updated evaluation loop with context extraction
for test_row in rerank_dataset:
    inputs = {"messages": [HumanMessage(content=test_row.eval_sample.user_input)]}
    response = cohere_compiled_graph.invoke(inputs)
    
    # DEBUG: Print which tools were called
    print(f"\n=== Question: {test_row.eval_sample.user_input[:80]}... ===")
    for msg in response["messages"]:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tool_call in msg.tool_calls:
                print(f"Tool called: {tool_call.get('name', 'unknown')}")
        if hasattr(msg, 'name') and msg.name:
            print(f"Tool response from: {msg.name}")
    
    # Extract the response text from the last message
    test_row.eval_sample.response = response["messages"][-1].content
    
    # Extract contexts from tool calls in the conversation
    retrieved_contexts = get_contexts_from_graph_response(response)
    
    # Store contexts for evaluation
    test_row.eval_sample.retrieved_contexts = retrieved_contexts
    
    print(f"Retrieved {len(retrieved_contexts)} contexts")
    print("-" * 50)
    time.sleep(2)


=== Question: What Acemoglu 2024 say about AI effect on economy?... ===
Tool called: arxiv
Tool response from: arxiv
Tool called: tavily_search_results_json
Tool response from: tavily_search_results_json

Processing ToolMessage from None

=== Starting message parsing ===
Message type: <class 'langchain_core.messages.human.HumanMessage'>
Error detecting tool type: 'NoneType' object has no attribute 'lower'
Detected tool type: None
Could not detect tool type

Processing ToolMessage from None

=== Starting message parsing ===
Message type: <class 'langchain_core.messages.ai.AIMessage'>
Detected tool type: arxiv
Extracted content type: <class 'str'>
Parsed 0 arXiv results

Processing ToolMessage from arxiv

=== Starting message parsing ===
Message type: <class 'langchain_core.messages.tool.ToolMessage'>
Detected tool type: arxiv
Extracted content type: <class 'str'>
Parsed 3 arXiv results

Processing ToolMessage from None

=== Starting message parsing ===
Message type: <class 'langchain_c

In [71]:
# Verify dataset is properly populated
df = rerank_dataset.to_pandas()
print("Dataset columns:", df.columns.tolist())
print("\nSample row check:")
sample = df.iloc[0]
print(f"- user_input: {type(sample['user_input'])} - {str(sample['user_input'])[:100]}...")
print(f"- response: {type(sample['response'])} - {str(sample['response'])[:100]}...")
print(f"- reference: {type(sample['reference'])} - {str(sample['reference'])[:100]}...")
print(f"- retrieved_contexts: {type(sample['retrieved_contexts'])} - Length: {len(sample['retrieved_contexts']) if sample['retrieved_contexts'] else 0}")

# Check if all required columns have data
print("\nData completeness check:")
print(f"- Rows with user_input: {df['user_input'].notna().sum()}/{len(df)}")
print(f"- Rows with response: {df['response'].notna().sum()}/{len(df)}")
print(f"- Rows with reference: {df['reference'].notna().sum()}/{len(df)}")
print(f"- Rows with retrieved_contexts: {df['retrieved_contexts'].notna().sum()}/{len(df)}")

Dataset columns: ['user_input', 'retrieved_contexts', 'reference_contexts', 'response', 'reference', 'synthesizer_name']

Sample row check:
- user_input: <class 'str'> - What Acemoglu 2024 say about AI effect on economy?...
- response: <class 'str'> - Daron Acemoglu's 2024 work on the economic effects of AI, especially as summarized in his paper "The...
- reference: <class 'str'> - Acemoglu (2024) discusses the intensified interest in the effects of artificial intelligence on econ...
- retrieved_contexts: <class 'list'> - Length: 8

Data completeness check:
- Rows with user_input: 10/10
- Rows with response: 10/10
- Rows with reference: 10/10
- Rows with retrieved_contexts: 10/10


In [72]:
# Run RAGAS evaluation with properly populated dataset
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig
from ragas import EvaluationDataset
from ragas.llms import LangchainLLMWrapper

# Create evaluation dataset from pandas
cohere_evaluation_dataset = EvaluationDataset.from_pandas(rerank_dataset.to_pandas())

# Configure evaluation
custom_run_config = RunConfig(timeout=360)
cohere_evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

# Run evaluation with all metrics
cohere_rerank_result = evaluate(
    dataset=cohere_evaluation_dataset,
    metrics=[
        LLMContextRecall(),      # Requires: user_input, retrieved_contexts, reference
        Faithfulness(),          # Requires: user_input, response, retrieved_contexts
        FactualCorrectness(),    # Requires: user_input, response, reference
        ResponseRelevancy(),     # Requires: user_input, response
        ContextEntityRecall(),   # Requires: user_input, retrieved_contexts, reference
        NoiseSensitivity()       # Requires: user_input, retrieved_contexts
    ],
    llm=cohere_evaluator_llm,
    run_config=custom_run_config
)

# Display results
print("RAGAS Evaluation Results:")
print(cohere_rerank_result)

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Exception raised in Job[39]: InternalServerError(<!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>


<title>api.openai.com | 520: Web server is returning an unknown error</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />


</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">
            <h1 class="inline-block sm:bl

RAGAS Evaluation Results:
{'context_recall': 0.7298, 'faithfulness': 0.9502, 'factual_correctness': 0.5200, 'answer_relevancy': 0.9410, 'context_entity_recall': 0.2951, 'noise_sensitivity_relevant': 0.4080}


In [73]:
print(cohere_rerank_result)

{'context_recall': 0.7298, 'faithfulness': 0.9502, 'factual_correctness': 0.5200, 'answer_relevancy': 0.9410, 'context_entity_recall': 0.2951, 'noise_sensitivity_relevant': 0.4080}


In [67]:
print(baseline_result)

{'context_recall': 0.5893, 'faithfulness': 0.9107, 'factual_correctness': 0.4860, 'answer_relevancy': 0.9497, 'context_entity_recall': 0.3454, 'noise_sensitivity_relevant': 0.3869}
